For this assignment my approach was to use a decision tree model with added features and nested cross-validation. The features that were added are features that have to do with whether balances and transactions add up correctly since that may be a pretty strong indicator of fraud. These features make it much easier for the tree model to make decisions since it does not require the tree itself to come up with the connection between the separate features. Then using nested cross-validation, the inner loop picks the `max_depth` and decision threshold and the outer loop gives an estimate of the f1 score on data that was not seen during tuning. At the end of the nested cross-validation the probability cutoff is chosen to maximize the f1 score instead of the default 0.5 probability cutoff in trees. Then lastly the model is refit on all of the training data, then using the threshold decided in the last part it predicts the outcome of the test file data and saves the results as `HW2_test_output.csv`. 

In [46]:
from sklearn import tree #model
from sklearn.metrics import f1_score #evaluation
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict #tuning
from sklearn.compose import ColumnTransformer #preprocessing
from sklearn.preprocessing import OneHotEncoder #preprocessing
from sklearn.pipeline import Pipeline #preprocessing
import numpy as np #utility
import pandas as pd #utility
#Import the necessary packages

In [47]:
train_data = pd.read_csv("HW2_training.csv")
print(train_data.head()) #read in training data and print the first few rows to view features

       type     amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0  CASH_OUT   94865.94       41338.00            0.00             0.0   
1   CASH_IN  318248.86        2980.00       321228.86        336116.4   
2   PAYMENT   17546.87           0.00            0.00             0.0   
3  CASH_OUT  126683.04       11162.85            0.00       2519453.7   
4  CASH_OUT  150054.00         356.00            0.00             0.0   

   newbalanceDest  isFraud  
0        94865.94        0  
1        17867.54        0  
2            0.00        0  
3      2646136.74        0  
4       150054.00        0  


In [48]:
#add new balance features to a copy of the data and return the copy with these new features
def add_balance_feats(df):
    df = df.copy()
    #How far off each side's balance is from being consistent (should = 0)
    df["orig_balance_error"] = (df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"])
    df["dest_balance_error"] = (df["oldbalanceDest"] + df["amount"] -df["newbalanceDest"])

    #Same errors above but scaled to the transaction size and clipped so there is no division by 0
    scale = df["amount"].clip(lower=1)
    df["orig_balance_error_scaled"] = df["orig_balance_error"] / scale
    df["dest_balance_error_scaled"] = df["dest_balance_error"] / scale

    #Binary flags for suspicious balances (empty accounts, unchanged destination balances)
    df["orig_balance_was_zero"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["orig_balance_is_zero"] = (df["newbalanceOrig"] == 0).astype(int)
    df["dest_balance_was_zero"] = (df["oldbalanceDest"] == 0).astype(int)
    df["dest_balance_unchanged"] = (df["oldbalanceDest"] == df["newbalanceDest"]).astype(int)

    return df

In [49]:
#add the balance features to the training data, then separate predictors from the target value
train_data = add_balance_feats(train_data)
X = train_data.drop(columns=["isFraud"]) #predictors
y = train_data["isFraud"] #target value
print(X.head()) #check if new features were added

       type     amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0  CASH_OUT   94865.94       41338.00            0.00             0.0   
1   CASH_IN  318248.86        2980.00       321228.86        336116.4   
2   PAYMENT   17546.87           0.00            0.00             0.0   
3  CASH_OUT  126683.04       11162.85            0.00       2519453.7   
4  CASH_OUT  150054.00         356.00            0.00             0.0   

   newbalanceDest  orig_balance_error  dest_balance_error  \
0        94865.94           -53527.94                0.00   
1        17867.54          -636497.72           636497.72   
2            0.00           -17546.87            17546.87   
3      2646136.74          -115520.19                0.00   
4       150054.00          -149698.00                0.00   

   orig_balance_error_scaled  dest_balance_error_scaled  \
0                  -0.564248                        0.0   
1                  -2.000000                        2.0   
2                

Numeric columns are just passed through normally, trees split on thresholds so scaling the numerical features is not necessary. The categorical columns are handled with one-hot encoding with `handle_unknown` set to ignore so that if the model sees a new category it does not break. Lastly `StratifiedKFold` is necessary since it keeps the ratio of fraud and non-fraud samples in folds the same for every fold. Frauds are very rare so it matters to keep the ratio the same so that there are not any folds without fraud examples and nothing to compare against.

In [50]:
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

#pass the numerical columns through without scaling and one-hot encode the categorical columns
preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

#StratifiedKFold outer loop
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state= 83)
outer_f1_scores = []

Through multiple different runs of `GridSearchCV` with different parameters being tested, the best performing parameters have already been added to the tree classifier. The best performing `max_depth` seems to change quite a lot between runs, so instead of the other values that are almost always the best at the same value, `max_depth` is chosen through cross-validation rather than prechosen. 
Fixed parameters:
- `Criterion="gini"`: Standard split quality measure.
- `ccp_alpha=1e-5`: Very light cost-complexity pruning to trim any splits that barely help.
- `min_samples_leaf=1`, `min_samples_split=2`: Default restrictions on leaf size, smaller leaves allow model to pick up on slight changes. 
- `class_weight=None`: class imbalance is handled by the decision threshold tuning rather than reweighting. 

In [51]:
#Only searching for the best performing max_depth
param_grid = {
    "classifier__max_depth": [10, 20, 50, None]}

tree_classifier = tree.DecisionTreeClassifier(criterion="gini", ccp_alpha=1e-5,
                        min_samples_leaf=1, min_samples_split=2, random_state=83, class_weight=None)


model = Pipeline(steps=[("preprocess", preprocessor), ("classifier", tree_classifier)])
#Look for threshold in 0.001-1.000 
thresholds = np.linspace(0, 1, 1001)

Start the nested cross-validation to tune the `max_depth` and threshold. These must be chosen without looking at the data that is scored on, since that would inflate the f1 score. So for each of the outer folds in `cv_outer`:
1. Split the outer fold into part training and part validation
2. Inner loop: `GridSearchCV` on the outer training part picks the `max_depth`
3. Get the out-of-fold fraud probabilities on the outer training part(`cross_val_predict`), so each probability comes from a model that did not see that row. Then choose the threshold which maximizes the f1 score.
4. Score the outer validation part once using the chosen `max_depth` and threshold to get honest f1 score for the fold.

In [52]:
for fold, (train_idx, val_idx) in enumerate(cv_outer.split(X, y), start=1):
    print(f"Starting out fold {fold}/5", flush=True)

    #Outer fold split into training and validation
    X_train_out = X.iloc[train_idx]
    y_train_out = y.iloc[train_idx]
    X_val_out = X.iloc[val_idx]
    y_val_out = y.iloc[val_idx]

    #Inner loop, choose the max_depth using only the outer training data (X_train_out)
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=83)
    search = GridSearchCV(model, param_grid, scoring="f1", cv=cv_inner, n_jobs=-1, verbose=2)
    search.fit(X_train_out, y_train_out)
    print("Grid search finished")

    #Calculate the out-of-fold probabilities on the outer training data, then pick threshold maximizing f1
    out_fold_probs = cross_val_predict(search.best_estimator_, X_train_out, 
                            y_train_out, cv=cv_inner, method="predict_proba", n_jobs=-1, verbose=2)[:, 1]
    print(f"Cross-validation predictions finished")
    #For each threshold calculate f1 score
    threshold_f1s = [f1_score(y_train_out, out_fold_probs >= threshold) 
                     for threshold in thresholds]
    best_thresh = thresholds[np.argmax(threshold_f1s)]#Save the highest f1 score threshold

    #Evaluate once on the held out outer fold.
    val_probs = search.best_estimator_.predict_proba(X_val_out)[:, 1]
    val_preds = val_probs >= best_thresh
    fold_f1 = f1_score(y_val_out, val_preds)
    outer_f1_scores.append(fold_f1)

    print("Best parameters:", search.best_params_)
    print(f"Fold {fold}: threshold={best_thresh:.4f}, F1={fold_f1:.6f}")

Starting out fold 1/5
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Grid search finished


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    3.9s remaining:    2.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.9s finished


Cross-validation predictions finished
Best parameters: {'classifier__max_depth': 10}
Fold 1: threshold=0.0010, F1=0.991736
Starting out fold 2/5
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Grid search finished


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    3.3s remaining:    2.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.5s finished


Cross-validation predictions finished
Best parameters: {'classifier__max_depth': 20}
Fold 2: threshold=0.0010, F1=0.980392
Starting out fold 3/5
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Grid search finished


...

Cross-validation predictions finished
Best parameters: {'classifier__max_depth': 20}
Fold 3: threshold=0.0010, F1=0.988889
Starting out fold 4/5
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Grid search finished


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    3.9s remaining:    2.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.9s finished


Cross-validation predictions finished
Best parameters: {'classifier__max_depth': 10}
Fold 4: threshold=0.0010, F1=0.994475
Starting out fold 5/5
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Grid search finished


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    5.0s remaining:    3.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    5.3s finished


Cross-validation predictions finished
Best parameters: {'classifier__max_depth': 10}
Fold 5: threshold=0.0010, F1=0.994505


The outer fold f1 scores have a mean of about 0.9899 with a standard deviation around 0.0052, so the performance of the model is stable across multiple folds. 

The chosen `max_depth` in the overall case is 10, but in some folds it is 20 which is why it is the only parameter other than the threshold being chosen by the nested cross-validation.

In this case every fold selected the threshold 0.001 which is very low. A deep tree's leaves are mostly only holding a single class especially since `min_samples_leaf=1`. So the predicted probabilities are close to 0 or 1, and in that case the threshold search adds little change to the f1 score.

In [53]:
print("Nested CV mean F1:", np.mean(outer_f1_scores))
print("Nested CV F1 standard deviation", np.std(outer_f1_scores))
cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=83)

final_search = GridSearchCV(model, param_grid, scoring="f1", cv=cv_final, n_jobs=-1, verbose=2)
final_search.fit(X, y)
print("Final best parameters:", final_search.best_params_)

Nested CV mean F1: 0.9899994431137517
Nested CV F1 standard deviation 0.005232277656715942
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Final best parameters: {'classifier__max_depth': 10}


Now for the final predictions on the actual test set:
1. Get the out-of-fold probabilities for the full training set and pick the f1-maximizing threshold.
2. Add the same balancing features to the test file
3. Predict with the final model and apply the threshold to get 0 or 1 labels.
4. Save the prediction results to `HW2_test_output.csv` with one column, no index, same row order, and as integers.

In [ ]:
#Threshold selection from out-of-fold probabilities on the full training set
oof_probs = cross_val_predict(final_search.best_estimator_, X, y, cv=cv_final, method="predict_proba"
                              , n_jobs=-1, verbose=2)[:, 1]

threshold_f1s = [f1_score(y, oof_probs >= threshold) for threshold in thresholds]
best_threshold = thresholds[np.argmax(threshold_f1s)]
print("Selected threshold:", best_threshold)

#Add the same balancing features and ensure the test data has the same columns as training
test_data = add_balance_feats(pd.read_csv("HW2_test_input.csv"))
X_test = test_data[X.columns]

#predict probabilities then convert to 0/1 labels
test_probs = final_search.best_estimator_.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

assert len(test_preds) == len(test_data), 'Prediction count does not match test rows'

#save the results to HW2_test_output.csv with no header and no index
pd.DataFrame({"isFraud": test_preds}).to_csv("HW2_test_output.csv", index=False, header=False)
print("Saved predictions to HW2_test_output.csv")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    5.2s remaining:    3.4s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    5.3s finished


Selected threshold: 0.001
Saved predictions to HW2_test_output.csv
